<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/resnet_robertith_ca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install albumentations scikit-learn -q

import os
import cv2
import glob
import copy
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import resnet34, ResNet34_Weights
import albumentations as A

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

BATCH_SIZE = 8
EPOCHS = 100
LR = 1e-4
IMG_SIZE = 256
SEED = 42
N_SPLITS = 5 # 5-Fold Cross Validation
CLASSES = ["benign", "malignant"]

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True

possible_paths = glob.glob("/kaggle/input/**/benign", recursive=True)
if not possible_paths:
    raise FileNotFoundError("Dataset not found!")
BASE_DIR = os.path.dirname(possible_paths[0])


train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

class BUSIDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform
        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            if not os.path.exists(cls_dir): continue
            images = [f for f in os.listdir(cls_dir) if f.endswith(".png") and "_mask" not in f]
            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = [f for f in os.listdir(cls_dir) if f.startswith(base_name + "_mask") and f.endswith(".png")]
                if not mask_files: continue
                self.samples.append((img_path, [os.path.join(cls_dir, f) for f in mask_files]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            combined_mask = np.logical_or(combined_mask, (mask > 0).astype(np.uint8))

        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=combined_mask)
            image, combined_mask = augmented["image"], augmented["mask"]

        combined_mask = (combined_mask > 0.5).astype(np.float32)
        return torch.from_numpy(image).permute(2, 0, 1).float(), torch.from_numpy(combined_mask).unsqueeze(0).float()

full_dataset = BUSIDataset(BASE_DIR, classes=CLASSES, transform=None)

class RobertsEdgeOperator(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[[[1.0, 0.0], [0.0, -1.0]]]])
        ky = torch.tensor([[[[0.0, 1.0], [-1.0, 0.0]]]])
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def forward(self, x):
        gray = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3]
        gray_padded = F.pad(gray, (0, 1, 0, 1), mode='replicate')
        edge = torch.sqrt(F.conv2d(gray_padded, self.kx)**2 + F.conv2d(gray_padded, self.ky)**2 + 1e-8)
        return edge

class DAUNet_SWA(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.q_conv = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.k_conv = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.v_conv = nn.Conv2d(in_channels, in_channels, 1)
        self.attn = nn.Conv2d((in_channels // 8) * 2 + in_channels, in_channels, 1)

    def forward(self, x):
        q, k, v = self.q_conv(x), self.k_conv(x), self.v_conv(x)
        A = torch.sigmoid(self.attn(torch.cat([q, k, v], dim=1)))
        return x + (v * A)

class PCBAM_Filter(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.cam_dense = nn.Sequential(
            nn.Linear(in_channels, in_channels // 8), nn.ReLU(),
            nn.Linear(in_channels // 8, in_channels)
        )
        self.sam_conv = nn.Conv2d(2, 1, 7, padding=3)

    def forward(self, x):
        b, c, _, _ = x.size()
        y_avg = self.cam_dense(F.adaptive_avg_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        y_max = self.cam_dense(F.adaptive_max_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        x_c = x * torch.sigmoid(y_avg + y_max)
        y_spat = torch.cat([torch.mean(x_c, dim=1, keepdim=True), torch.max(x_c, dim=1, keepdim=True)[0]], dim=1)
        return x_c * torch.sigmoid(self.sam_conv(y_spat))

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)


class MaxDiceUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberts = RobertsEdgeOperator()

        # Load Pretrained ResNet34
        resnet = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)

        old_conv = resnet.conv1
        self.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.conv1.weight[:, :3] = old_conv.weight
            self.conv1.weight[:, 3] = old_conv.weight[:, 0]

        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool

        # Encoder Stages
        self.layer1 = resnet.layer1 # 64 channels
        self.layer2 = resnet.layer2 # 128 channels
        self.layer3 = resnet.layer3 # 256 channels
        self.layer4 = resnet.layer4 # 512 channels

        self.pcbam1 = PCBAM_Filter(64)
        self.pcbam2 = PCBAM_Filter(128)
        self.pcbam3 = PCBAM_Filter(256)
        self.pcbam4 = PCBAM_Filter(512)

        # THE FIX: Added pooling before the bottleneck
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.swa = DAUNet_SWA(1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # THE FIX: Extra upsampling stages to get back to 256x256
        self.up0 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec0 = DoubleConv(128, 64) # 64 from up0 + 64 from x0 skip connection

        self.up_out = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec_out = DoubleConv(32, 32)
        self.final = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        # Early Fusion
        x_edge = self.roberts(x)
        x_fused = torch.cat([x, x_edge], dim=1)

        # ResNet Encoder
        x0 = self.relu(self.bn1(self.conv1(x_fused))) # 128x128
        x1 = self.maxpool(x0) # 64x64

        e1 = self.layer1(x1) # 64x64
        e2 = self.layer2(e1) # 32x32
        e3 = self.layer3(e2) # 16x16
        e4 = self.layer4(e3) # 8x8

        s1 = self.pcbam1(e1)
        s2 = self.pcbam2(e2)
        s3 = self.pcbam3(e3)
        s4 = self.pcbam4(e4)

        # THE FIX: Apply pooling so the bottleneck is 4x4
        b = self.swa(self.bottleneck(self.pool(e4)))

        # Decoder (Math perfectly aligns now)
        d4 = self.dec4(torch.cat([self.up4(b), s4], dim=1)) # 8x8
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1)) # 16x16
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1)) # 32x32
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1)) # 64x64

        # Final Upsample using x0 as an extra skip connection
        d0 = self.dec0(torch.cat([self.up0(d1), x0], dim=1)) # 128x128
        d_out = self.dec_out(self.up_out(d0)) # 256x256

        return self.final(d_out)
# ==========================================
# 5. HYBRID LOSS & METRICS
# ==========================================
class HybridLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        probs = torch.sigmoid(logits).view(-1)
        targets_f = targets.view(-1)

        # Dice Loss
        inter = (probs * targets_f).sum()
        dice = 1 - (2. * inter + self.smooth) / (probs.sum() + targets_f.sum() + self.smooth)

        # Focal Loss
        pt = torch.where(targets_f == 1, probs, 1 - probs)
        focal = -((1 - pt) ** 2) * torch.log(pt + 1e-8)
        focal = focal.mean()

        return 0.4 * bce + 0.4 * dice + 0.2 * focal

def calc_dice(y_pred, y_true, smooth=1e-5):
    y_pred = (torch.sigmoid(y_pred) > 0.5).float().view(-1)
    y_true_f = y_true.view(-1)
    inter = (y_true_f * y_pred).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred.sum() + smooth)

indices = np.arange(len(full_dataset))
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
criterion = HybridLoss()

fold_models = []

print(f"\n STARTING {N_SPLITS}-FOLD CROSS VALIDATION")
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(kf.split(indices)):
    print(f"\n--- FOLD {fold+1}/{N_SPLITS} ---")

    train_sub = Subset(BUSIDataset(BASE_DIR, CLASSES, train_transform), train_idx)
    val_sub = Subset(BUSIDataset(BASE_DIR, CLASSES, val_test_transform), val_idx)

    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = MaxDiceUNet().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda')

    best_val_dice = 0.0
    best_weights = None

    for epoch in range(EPOCHS):
        model.train()
        train_dice = 0

        for images, masks in tqdm(train_loader, desc=f"F{fold+1} E{epoch+1}", leave=False):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                train_dice += calc_dice(logits, masks).item()

        scheduler.step()

        # Validation
        model.eval()
        val_dice = 0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                val_dice += calc_dice(logits, masks).item()

        avg_v_dice = val_dice / len(val_loader)

        if avg_v_dice > best_val_dice:
            best_val_dice = avg_v_dice
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, f"/kaggle/working/best_model_fold_{fold+1}.pth")

    print(f" Fold {fold+1} Complete | Best Val Dice: {best_val_dice:.4f}")
    fold_models.append(f"/kaggle/working/best_model_fold_{fold+1}.pth")



def predict_tta(model, image_tensor):
    """ Passes original, h-flip, and v-flip versions, then averages. """
    p1 = torch.sigmoid(model(image_tensor)) # Original
    p2 = torch.sigmoid(model(torch.flip(image_tensor, dims=[3]))) # H-Flip
    p3 = torch.sigmoid(model(torch.flip(image_tensor, dims=[2]))) # V-Flip


    p2_unflip = torch.flip(p2, dims=[3])
    p3_unflip = torch.flip(p3, dims=[2])

    return (p1 + p2_unflip + p3_unflip) / 3.0

# Load all 5 trained models into memory
ensemble = []
for path in fold_models:
    m = MaxDiceUNet().to(device)
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    ensemble.append(m)


eval_loader = DataLoader(BUSIDataset(BASE_DIR, CLASSES, val_test_transform), batch_size=4, shuffle=False)

total_ensemble_dice = 0
with torch.no_grad():
    for images, masks in tqdm(eval_loader, desc="TTA Ensembling"):
        images, masks = images.to(device), masks.to(device)

        ensemble_preds = torch.zeros_like(masks).to(device)


        for m in ensemble:
            with torch.amp.autocast('cuda'):
                ensemble_preds += predict_tta(m, images)


        ensemble_preds = ensemble_preds / len(ensemble)


        ensemble_preds_bin = (ensemble_preds > 0.5).float()

        inter = (masks.view(-1) * ensemble_preds_bin.view(-1)).sum()
        dice = (2. * inter + 1e-5) / (masks.sum() + ensemble_preds_bin.sum() + 1e-5)
        total_ensemble_dice += dice.item()

final_dice = total_ensemble_dice / len(eval_loader)

print("\n" + "=" * 60)
print(f" TEST  DICE SCORE: {final_dice:.4f}")
print("=" * 60)

Using device: cuda

 STARTING 5-FOLD CROSS VALIDATION

--- FOLD 1/5 ---


 Fold 1 Complete | Best Val Dice: 0.7914

--- FOLD 2/5 ---


 Fold 2 Complete | Best Val Dice: 0.4652

--- FOLD 3/5 ---


 Fold 3 Complete | Best Val Dice: 0.8174

--- FOLD 4/5 ---


 Fold 4 Complete | Best Val Dice: 0.8756

--- FOLD 5/5 ---


 Fold 5 Complete | Best Val Dice: 0.7790


TTA Ensembling: 100%|██████████| 162/162 [00:41<00:00,  3.88it/s]


 TEST  DICE SCORE: 0.9192
